# 9. AIND Ephys QC Collector

Build an `AINDEPhysQCCollectorScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysQCCollectorTask` for each coordinate. The task itself clones [aind-ephys-qc-collector](https://github.com/AllenNeuralDynamics/aind-ephys-qc-collector) on first use, seeds its `data/` with the per-recording `quality_control_<name>.json` + `quality_control_<name>/` figure folders from notebook 08, runs `code/run_capsule.py`, and copies the aggregated `quality_control.json` + flat `quality_control/<probe>/` figure tree into the single config's `coordinate_output_root` (`obi-output/09_aind_ephys_qc_collector/grid_scan/0/`).

## Imports and prerequisites

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import obi_one as obi

In [2]:
subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "aind-data-schema"],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Resolved 14 packages in 7ms
Uninstalled 2 packages in 8ms
Installed 2 packages in 2ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema'], returncode=0)

## Build the scan config

In [3]:
qc_output_path = (
    Path.cwd() / "../../../obi-output/08_aind_ephys_processing_qc/grid_scan/0"
).resolve()
assert qc_output_path.exists(), (
    f"{qc_output_path} not found. Run 08_aind_ephys_processing_qc.ipynb first."
)

scan_config = obi.AINDEPhysQCCollectorScanConfig(
    initialize=obi.AINDEPhysQCCollectorScanConfig.Initialize(
        qc_output_path=qc_output_path,
    ),
)

## Generate the grid scan and run the QC-collector task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/09_aind_ephys_qc_collector/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 18:15:38,156] INFO: Seeded 1 QC recording(s) into /tmp/aind-ephys-qc-collector/data


[2026-04-29 18:15:38,156] INFO: Running python -u run_capsule.py


[2026-04-29 18:15:38,492] INFO: 
EPHYS QC COLLECTION
Found 1 quality control files
All recording segments are the same. Dropping recording segment from recording name.
	Collected 5 metrics for 0 tags and 1 streams.
	Tags allowed to fail: []
EPHYS QC COLLECTION time: 0.0s



[PosixPath('../../../obi-output/09_aind_ephys_qc_collector/grid_scan/0')]

## Inspect the aggregated quality_control.json

In [5]:
coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

qc = json.loads((coord_dir / "quality_control.json").read_text())
print("schema_version:", qc.get("schema_version"))
print("default_grouping:", qc.get("default_grouping"))
print("allow_tag_failures:", qc.get("allow_tag_failures"))
print(f"Total metrics: {len(qc.get('metrics', []))}")
for m in qc.get("metrics", []):
    print(f"  {m.get('stage'):>10}  {m.get('name'):40}  reference={m.get('reference')}")

coordinate_output_root: ../../../obi-output/09_aind_ephys_qc_collector/grid_scan/0
schema_version: 2.4.1
default_grouping: ['probe', 'stage']
allow_tag_failures: []
Total metrics: 5
    Raw data  Raw data block0_None                      reference=quality_control/block0_None/traces_raw.png
    Raw data  PSD block0_None                           reference=quality_control/block0_None/psd.png
    Raw data  RMS block0_None                           reference=quality_control/block0_None/rms.png
  Processing  Unit Metrics Yield - block0_None          reference=quality_control/block0_None/unit_yield.png
  Processing  Firing rate - block0_None                 reference=quality_control/block0_None/firing_rate.png
